# Optional Project - Colab Part3 Training

Runs Part 3: muP LR sweep on Tiny, transfer the selected LR to all muP model sizes, compare SP vs muP scaling curves, and extrapolate validation loss for a 10x larger model. Re-run interrupted training cells to resume from Google Drive checkpoints.

In [1]:
# ===== User config =====
REPO_URL = "https://github.com/Peng-y-x/optionalproject.git"
REPO_DIR = "/content/optionalproject"
REPO_BRANCH = "run"
MUP_SWEEP_CONFIG = "configs/mup_sweep_lr.yaml"
MUP_BEST_LR_JSON = "outputs/part3_mup_lr_sweep/best_lr.json"
DRIVE_MUP_BEST_LR_JSON = "/content/drive/MyDrive/svg-scaling/part3_mup_lr_sweep/best_lr.json"
SP_RUNS_DIR = "outputs/part2_v2"
DRIVE_SP_RUNS_DIR = "/content/drive/MyDrive/svg-scaling/part2_v2"


In [2]:
# 1) Clone repo and checkout branch
import os
if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    print('Repo already exists:', REPO_DIR)
%cd $REPO_DIR
!git fetch origin
!git checkout {REPO_BRANCH}
!git pull origin {REPO_BRANCH}
!git branch --show-current
!git rev-parse --short HEAD


Cloning into '/content/optionalproject'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 151 (delta 85), reused 110 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 257.87 KiB | 392.00 KiB/s, done.
Resolving deltas: 100% (85/85), done.
/content/optionalproject
Already on 'run'
Your branch is up to date with 'origin/run'.
From https://github.com/Peng-y-x/optionalproject
 * branch            run        -> FETCH_HEAD
Already up to date.
run
583c1ea


In [3]:
# 2) Mount Google Drive for resumable checkpoints
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/svg-scaling/part3_mup
!mkdir -p /content/drive/MyDrive/svg-scaling/part3_mup_lr_sweep
!mkdir -p /content/drive/MyDrive/svg-scaling/part3_analysis


Mounted at /content/drive


In [4]:
# 3) Install system + Python dependencies
!apt-get update -y
!apt-get install -y libcairo2 libcairo2-dev libffi-dev
!python -m pip install --upgrade pip
!pip install -r requirements.txt


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]       
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease   
Get:13 https://r2u.stat.illinois.ed

In [ ]:
# 4) HF auth from Colab Keys (key name must be HF_TOKEN)
import os
from google.colab import userdata
token = userdata.get('HF_TOKEN')
if token:
    os.environ['HF_TOKEN'] = token
    print('HF token loaded from Colab key.')
else:
    print('HF token not found in Colab key HF_TOKEN. Public dataset loading may still work.')
print('has_hf_token:', bool(os.getenv('HF_TOKEN')))


In [6]:
# 5) GPU sanity check
import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('bf16_supported', torch.cuda.is_bf16_supported())


torch 2.10.0+cu128
cuda_available True
gpu NVIDIA A100-SXM4-80GB
bf16_supported True


In [7]:
# 6) Part 3 muP LR sweep on Tiny model
# If Colab disconnects, rerun this cell; each LR run resumes from Drive latest.pt.
%cd $REPO_DIR
!python scripts/run_mup_lr_sweep.py --config {MUP_SWEEP_CONFIG}


/content/optionalproject
[mup-sweep] lr=0.001 run=tiny_mup_lr_1.0e-03
/content/optionalproject/src/train/trainer.py:457: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.device.type == "cuda" and self.precision == "fp16")
README.md: 100% 654/654 [00:00<00:00, 3.24MB/s]
data/train-00000-of-00001.parquet: 100% 410M/410M [00:06<00:00, 65.8MB/s]
data/validation-00000-of-00001.parquet: 100% 4.11M/4.11M [00:00<00:00, 10.0MB/s]
data/test-00000-of-00001.parquet: 100% 4.25M/4.25M [00:00<00:00, 10.3MB/s]
Generating train split: 100% 155571/155571 [00:01<00:00, 78655.55 examples/s] 
Generating validation split: 100% 1595/1595 [00:00<00:00, 96031.16 examples/s]
Generating test split: 100% 1590/1590 [00:00<00:00, 108845.17 examples/s]
{"type": "train", "step": 20, "tokens_seen": 140138, "next_row_index": 214, "train_loss": 8.204076991125948, "lr": 3.025936599423631e-0

In [8]:
# 7) Inspect best muP LR
import json, shutil
from pathlib import Path
if not Path(MUP_BEST_LR_JSON).exists() and Path(DRIVE_MUP_BEST_LR_JSON).exists():
    Path(MUP_BEST_LR_JSON).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_MUP_BEST_LR_JSON, MUP_BEST_LR_JSON)
best = json.loads(Path(MUP_BEST_LR_JSON).read_text())
print(json.dumps(best, indent=2))
MUP_BEST_LR = best['learning_rate']
print('MUP_BEST_LR=', MUP_BEST_LR)


{
  "learning_rate": 0.004,
  "val_loss": 0.7586738941988432,
  "val_ppl": 2.135442519305405,
  "val_tokens": 1006621.0,
  "run_name": "tiny_mup_lr_4.0e-03",
  "global_step": 14612,
  "tokens_seen": 100470493,
  "num_parameters": 1579776,
  "num_parameters_non_embedding": 1055488,
  "wall_clock_seconds": 331.82196068763733,
  "tokens_per_second_epoch": 302784.33890208526,
  "peak_gpu_memory_gb": 1.3334436416625977
}
MUP_BEST_LR= 0.004


In [9]:
# 8) Train all five muP model sizes for exactly one epoch with the selected Tiny LR
# If Colab disconnects, rerun this cell; each model resumes from Drive latest.pt.
%cd $REPO_DIR
!python scripts/run_part3_mup_all.py --best-lr-json {MUP_BEST_LR_JSON}


/content/optionalproject
[part3-mup] /usr/bin/python3 scripts/run_mup_train.py --config configs/mup_tiny.yaml --learning-rate 0.004
/content/optionalproject/src/train/trainer.py:457: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.device.type == "cuda" and self.precision == "fp16")
{"type": "train", "step": 20, "tokens_seen": 140138, "next_row_index": 214, "train_loss": 8.007735475796773, "lr": 0.00012103746397694524, "tokens_per_second": 147908.22603951726, "gpu_memory_gb": 1.2932367324829102, "elapsed_seconds": 0.9485821723937988}
{"type": "train", "step": 40, "tokens_seen": 279764, "next_row_index": 435, "train_loss": 7.253137233156104, "lr": 0.0002363112391930836, "tokens_per_second": 342962.72563691274, "gpu_memory_gb": 1.3226532936096191, "elapsed_seconds": 1.3562757968902588}
{"type": "train", "step": 60, "tokens_seen": 417436, "next_row_index": 6

In [10]:
# 9) If Part 2 SP outputs are only on Drive, copy them locally for comparison
from pathlib import Path
if not list(Path(SP_RUNS_DIR).glob('*/final_metrics.json')) and Path(DRIVE_SP_RUNS_DIR).exists():
    !mkdir -p {SP_RUNS_DIR}
    !cp -r {DRIVE_SP_RUNS_DIR}/* {SP_RUNS_DIR}/
print('SP files:', len(list(Path(SP_RUNS_DIR).glob('*/final_metrics.json'))))
print('muP files:', len(list(Path('outputs/part3_mup').glob('*/final_metrics.json'))))


SP files: 10
muP files: 5


In [11]:
# 10) Fit SP and muP power laws, create comparison plot/table, and extrapolate 10x larger model
%cd $REPO_DIR
!python scripts/fit_part3_scaling.py \
  --sp-runs-dir {SP_RUNS_DIR} \
  --mup-runs-dir outputs/part3_mup \
  --sp-sweep-json outputs/part2_lr_sweep_v2/sweep_results.json \
  --mup-sweep-json outputs/part3_mup_lr_sweep/sweep_results.json \
  --output-dir outputs/part3_analysis \
  --drive-output-dir /content/drive/MyDrive/svg-scaling/part3_analysis


/content/optionalproject
{
  "sp_runs_dir": "outputs/part2_v2",
  "mup_runs_dir": "outputs/part3_mup",
  "sp_fit": {
    "a": 0.4745636624228082,
    "alpha": 1.6154817978376846e-16,
    "c": 0.5872712142702109,
    "covariance": [
      [
        3.5660699482330265,
        0.9148207692665254,
        3.5660699482330265
      ],
      [
        0.9148207692665254,
        0.23655380684889663,
        0.9148207692665254
      ],
      [
        3.5660699482330265,
        0.9148207692665254,
        3.5660699482330265
      ]
    ],
    "rmse": 0.47495817938933577
  },
  "mup_fit": {
    "a": 10215.16600265542,
    "alpha": 0.7607356250209367,
    "c": 0.567584400560452,
    "covariance": [
      [
        2252672081.4142337,
        15703.574510327966,
        1101.7100074877533
      ],
      [
        15703.574510327968,
        0.10956084491018157,
        0.007778273389808971
      ],
      [
        1101.7100074877533,
        0.0077782733898089694,
        0.0007315939424146015


In [12]:
# 11) Inspect key Part 3 outputs
from pathlib import Path
import json
for p in sorted(Path('outputs/part3_mup').glob('*/final_metrics.json')):
    m = json.loads(p.read_text())
    print(p.parent.name, {k: m.get(k) for k in ['num_parameters','val_loss','val_ppl','tokens_seen','wall_clock_seconds','peak_gpu_memory_gb']})
summary = Path('outputs/part3_analysis/part3_scaling_comparison.json')
if summary.exists():
    print(json.dumps(json.loads(summary.read_text()), indent=2))


large_mup {'num_parameters': 34670592, 'val_loss': 0.59233212218585, 'val_ppl': 1.8082004463808434, 'tokens_seen': 100470493, 'wall_clock_seconds': 1434.4846985340118, 'peak_gpu_memory_gb': 4.722039222717285}
medium_mup {'num_parameters': 13006848, 'val_loss': 0.6265052948461459, 'val_ppl': 1.871060336146737, 'tokens_seen': 100470493, 'wall_clock_seconds': 704.7880222797394, 'peak_gpu_memory_gb': 2.620593547821045}
small_mup {'num_parameters': 3849216, 'val_loss': 0.6491834512427738, 'val_ppl': 1.913977334951987, 'tokens_seen': 100470493, 'wall_clock_seconds': 517.3446805477142, 'peak_gpu_memory_gb': 1.845517635345459}
tiny_mup {'num_parameters': 1579776, 'val_loss': 0.7695060466724455, 'val_ppl': 2.158699693494306, 'tokens_seen': 100470493, 'wall_clock_seconds': 324.3431384563446, 'peak_gpu_memory_gb': 1.3334174156188965}
xl_mup {'num_parameters': 89774592, 'val_loss': 0.5675844005604521, 'val_ppl': 1.764000781409577, 'tokens_seen': 100470493, 'wall_clock_seconds': 2768.41366648674, '